# 08 — Faz 2: sensör sağlığı kapısı (tek oturum)
⚠️ **Test seti bu notebook'ta bir kez, 3 koşulda ölçülür; sonra eşik ya da kural değişmez.**
**Çalışma zamanı → A100** (GPU: Ultralytics val; 12 CPU çekirdeği: koşul üretimi ve
bootstrap). Secrets: `KAGGLE_API_TOKEN`, `GITHUB_TOKEN`. Oturum koparsa yeniden
**Tümünü çalıştır**: eşik dosyası ve biten koşullar atlanır.
Süreler tahmindir (Faz 1'de E6 koşul başına ≈ 1 dk). Bütün çıktıları Claude'a getir.

In [ ]:
BRANCH = 'f2'
DRIVE = '/content/drive/MyDrive/baykus'
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
print('token yüklendi (değer yazdırılmadı)')

In [ ]:
# Repo private ise Secrets'a GITHUB_TOKEN ekle (fine-grained, sadece bu repo, 'Contents: read').
import os
from google.colab import userdata
try:
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = f'https://{tok}@github.com/OgzhnTarhn/baykus.git' if tok else 'https://github.com/OgzhnTarhn/baykus.git'
if not os.path.isdir('/content/baykus/.git'):
    !git clone -q {url} /content/baykus
!git -C /content/baykus fetch -q {url} {BRANCH}
!git -C /content/baykus checkout -q -B {BRANCH} FETCH_HEAD
!git -C /content/baykus remote set-url origin https://github.com/OgzhnTarhn/baykus.git
%cd /content/baykus
!git log --oneline -3

In [ ]:
%%time
!pip -q install uv kaggle
!uv pip install --system -q -e '/content/baykus/python[train]'
import sys
sys.path.insert(0, '/content/baykus/python/src')
!python --version && nproc && (nvidia-smi -L || true)

In [ ]:
%%time
# İşlenmiş veri: private Kaggle veri seti (ADR 0004). Kaggle yüklenen tar'ları klasöre açtı;
# CLI klasör seçerek indiremediği için veri setinin tamamı iner (~9,8 GB).
DS = 'ouzhantarhan/baykus-dv-rgbt'
!kaggle datasets download {DS} -p /content/dv_rgbt --unzip -q
!mv /content/dv_rgbt/meta /content/meta
!baykus-data illum-relabel --meta /content/meta   # ADR 0005: Kaggle'daki etiketler eski eşikle
!baykus-data slices --out /content/dv_rgbt --meta /content/meta
!baykus-data yaml --out /content/dv_rgbt
!du -sh /content/dv_rgbt/* /content/meta

In [ ]:
# Model ve Faz 1 taban tahminleri (kapısız E3c; val, 16 koşul). Eksik varsa dur.
import json, os
from pathlib import Path
from baykus.robustness.corruptions import suite_conditions
from baykus.train.checkpoints import find_runs
NPROC = os.cpu_count()
EXPECT = '2026-10-15_e3c-modality-dropout_s_d4e1280'   # results/robust'taki E3c kayıtları
done = find_runs(f'{DRIVE}/runs', 'e3c_modality_dropout', finished=True)
assert done, 'E3c tam eğitimi bulunamadı'
RUN = json.loads((done[-1] / 'baykus_train.json').read_text(encoding='utf-8'))['run_id']
print('E3c →', RUN)
assert RUN == EXPECT, f"run kimliği {RUN} ≠ {EXPECT}: Claude'a söyle"
MODEL = f'../configs/e3c_modality_dropout.yaml={done[-1]}/weights/best.pt'
missing = [c.id for c in suite_conditions()
           if not Path(f'{DRIVE}/robust/{RUN}_val_{c.id}/predictions_coco.json.gz').exists()]
print('eksik Faz 1 taban tahmini:', missing or 'yok')
assert not missing, "eksik koşulları Claude'a söyle (03_robust ile yeniden üretilir)"
TH = f'{DRIVE}/results/health/thresholds.json'
COMMON = f'--data-root /content/dv_rgbt --meta /content/meta --project {DRIVE}/robust --results {DRIVE}/results --workers {NPROC}'
TEST_CONDS = 'clean noise-20 misalign-20'

In [ ]:
%%time
# 1) Eşikler: train'den 500 görüntü (seed 42) × 16 koşul, Youden J (≈ 2-4 dk).
# Dosya varsa yeniden üretilmez: değişirse biten koşullar "başka eşik" hatası verir.
if Path(TH).exists():
    print('eşikler zaten var:', TH)
else:
    !cd /content/baykus/python && baykus-health fit --data-root /content/dv_rgbt --meta /content/meta --out {TH} --features-out {DRIVE}/health/train_features.csv --workers {NPROC}
print(json.dumps({k: round(v['threshold'], 3) for k, v in json.load(open(TH))['states'].items()}))

In [ ]:
%%time
# 2) Val, kapılı sağlamlık paketi: 16 koşul; yalnız kapının piksel değiştirdiği koşullar
# yeniden değerlendirilir (≈ 15-25 dk). Son 20 satırı getir.
!cd /content/baykus/python && baykus-health suite --model {MODEL} --thresholds {TH} --split val --work /content/health_work {COMMON}

In [ ]:
%%time
# 3) Test, taban (kapısız E3c): temiz, IR gürültüsü σ 20, hizasızlık 20 px (≈ 12-18 dk).
!cd /content/baykus/python && baykus-robust run --model {MODEL} --split test --conditions {TEST_CONDS} --n-boot 0 --work /content/robust_work {COMMON}

In [ ]:
%%time
# 4) Test, kapılı: aynı 3 koşul (≈ 10-15 dk).
!cd /content/baykus/python && baykus-health suite --model {MODEL} --thresholds {TH} --split test --conditions {TEST_CONDS} --work /content/health_work {COMMON}

In [ ]:
%%time
# 5) E3c + kapı − E3c: 1000 eşleştirilmiş bootstrap (val ≈ 3-6 dk, test ≈ 5-10 dk).
# Kabul satırlarını getir.
for SPLIT in ['val', 'test']:
    !cd /content/baykus/python && baykus-health compare --run-id {RUN} --split {SPLIT} --robust-preds {DRIVE}/robust --data-root /content/dv_rgbt --results {DRIVE}/results --workers {NPROC}

In [ ]:
# 6) İndirilecekler: sonuç klasörleri ve laptopa inecek ONNX test tahminlerinin boyutu.
!du -sh {DRIVE}/results/health {DRIVE}/results/gate
!ls {DRIVE}/results/robust | grep _test_
ONNX_TEST = f'{DRIVE}/edge/eval/{RUN}_test_onnx_fp32/predictions_coco.json.gz'
!ls -la "{ONNX_TEST}"